# arrodeio-llm — Phase 2 (SFT) on a free GPU

This trains `scripts/sft.py` (the exact same file that's in the repo) on Colab's
free T4 GPU. Takes about 10 minutes. At the end you download one ~9 MB file and
drop it into your laptop's `model/` folder.

**Before running anything:** menu **Runtime → Change runtime type → T4 GPU → Save**.

Then run each cell in order with the ▶ button (or **Runtime → Run all**).

### 1. Check we actually got a GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

If that errors or says "command not found", you didn't switch to a GPU runtime —
go back and do the Runtime menu step above.

### 2. Get the code and install the libraries

In [ ]:
%cd /content
!rm -rf arrodeio-llm && git clone --depth 1 https://github.com/bsana1/arrodeio-llm.git
%cd /content/arrodeio-llm
!pip install -q -U transformers peft accelerate
# Colab ships an old `torchao` that makes peft's LoRA setup crash; we don't use it.
!pip uninstall -q -y torchao

### 3. Build the training data

`make_sft_data.py` turns the real cordel stanzas in `data/cordel/` into ~600
instruction/response pairs (see the script for how). The 50 hand-written
`mote → glosa` pairs in `data/sft.jsonl` are already in the repo.

In [ ]:
!python scripts/make_sft_data.py
!wc -l data/sft.jsonl data/sft_synth.jsonl

### 4. Train (~10 minutes)

Watch the `loss` number drop and the `›` sample lines shift from chatty prose
toward cordel. This is Phase 2 happening.

In [ ]:
!python scripts/sft.py

### 5. Download the adapter

This is the only thing you keep — a small file with just the LoRA weights, not
the whole model.

In [ ]:
import shutil
shutil.make_archive('/content/cordel-sft-lora', 'zip', 'model/cordel-sft-lora')
from google.colab import files
files.download('/content/cordel-sft-lora.zip')

### 6. Back on your laptop

```bash
cd ~/src/arrodeio-llm
unzip ~/Downloads/cordel-sft-lora.zip -d model/cordel-sft-lora
./venv/bin/python scripts/chat.py            # talk to the fine-tuned model
./venv/bin/python scripts/chat.py --base     # talk to raw Tucano, to compare
```

Inference on your CPU is slow (~1 min per answer) but it works. The training —
the part that needed the GPU — is done.